# Post-quantum SSH, end to end
## Containers, router CLI, Ansible

Is the SSH session you manage your routers over post-quantum? Probably not yet.
This notebook shows what it takes: OpenSSH in containers first, where hybrid
ML-KEM is the default, then one line on a real IOS XE 26.2 router, checked from
your own laptop, then the same line on every router with Ansible.

**First time?** Build your lab with [`demos/README.md`](../README.md)
(prerequisites, credentials). SSH only needs each router's management address:
no underlay, no tunnels. This notebook checks the rest in
[Before each run](#preflight).

Four kinds of cell:

- **Act 1 cells** drive OpenSSH in containers and draw the handshake as it happens.
- **Laptop probes** (`ssh_kex_probe(...)`) open an SSH connection from *your* laptop,
  stop right after the key exchange (no password is ever sent) and report what was
  agreed.
- **Router checks** (`r1(...)`, `check()`) run read-only `show` commands through
  Ansible. In Act 2 you type every change yourself, and the checks confirm you
  got the result. The one exception is `setup()`, which configures and says so.
- **Playbook cells** (`playbook(...)`) run Act 3 from `deploy/c8000/automation`
  with its own venv and keep the `PLAY RECAP` visible.

## Contents

- [Setup](#setup)
- [Before each run](#preflight): containers, routers, automation
- [Act 1: containers](#act1): Demo 1
- [Starting at Act 2?](#act2-setup): set up the routers without the containers
- [Act 2: the router's SSH server, from your laptop](#act2): Demos 2 to 4
- [Act 3: Ansible](#act3): Demo 5
- [Clean up](#after)

<a id="setup"></a>
## Setup

Run this once per kernel. It finds `pqc_demo.py` in `demos/containers_demo/`, the
shared helpers in `demos/common/` and `ssh_lab.py` next to this notebook,
whichever folder Jupyter started in.

In [ ]:
import sys, pathlib, re
_here = pathlib.Path.cwd()
_demos = next((p for p in (_here.parent, _here, _here / 'demos') if (p / 'common' / 'lab.py').exists()), None)
if _demos is None:
    raise RuntimeError('Start Jupyter inside the repo (see demos/README.md) so the helpers can be found.')
for _p in ('containers_demo', 'common', 'ssh_demo'):
    sys.path.insert(0, str(_demos / _p))
from pqc_demo import *  # noqa: F401,F403  containers, panels
from ssh_lab import *   # noqa: F401,F403  routers, playbooks, probes, labs

# Ladder for the Act 1 panels (same object the learner notebook uses).
ssh = SSHDemo()

set_presenter_mode(True)
print('Repo:       ', REPO_ROOT)
print('Automation: ', AUTOMATION_DIR)
print('Containers:  ssh_server ssh_client | host')
print('Routers:     r1 r2 r3 (read-only)   Laptop: ssh_kex_probe()   Playbooks: playbook()')

---
<a id="preflight"></a>
## Before each run

Most failures are leftovers from the last run, so check everything before you
start. If a check fails, the fix is in the table below or in
the [one-time setup](../README.md).

### Containers

**First time on this machine only:** build this demo's labs. It compiles OpenSSH 10.4 from source, so give it a few minutes.

In [ ]:
# host: build the Act 1 images (first time on this machine)
labs.build()

In [ ]:
# host: bring this demo's stacks up, then show the readiness board
labs.up()

If anything in Act 1 sticks later, `labs.panic()` stops stray daemons and
captures without taking the stacks down.

The board below should be all green. No Docker? The container rows fail, and
that's fine if you start at Act 2.

In [ ]:
# host: Act 1 containers up, Act 2/3 prerequisites in place
labs.status()

The `ssh -Q kex` row is your laptop's OpenSSH. Acts 2 and 3 connect from here,
and a client older than OpenSSH 9.9 doesn't know `mlkem768x25519-sha256`.

### Routers: is the lab in Act 2's starting state?

The target: **every router runs SSH v2 with its platform default KEX list, which
has no ML-KEM in it, and no EC host key.** Why care about the host key? Because
of a lockout trap that `ssh.md` describes: import an EC keypair and the server
starts advertising an ECDSA host key it then can't use.

This cell only reads (about 15 seconds). Starting from scratch?
[`setup()`](#act2-setup) fixes the KEX rows.

In [ ]:
# R1, R2, R3: read-only checks of the Act 2 starting state
check()

**If a row fails**, here's where the fix lives:

| Row | Fix |
|---|---|
| SSH server | `ip ssh version 2`, plus a key: `crypto key generate rsa modulus 2048` |
| KEX line, KEX list | [`setup()`](#act2-setup), or `no ip ssh server algorithm kex` |
| EC host key | `ip ssh server algorithm hostkey rsa-sha2-512 rsa-sha2-256` pins the host key to RSA (see `deploy/c8000/ssh.md`, "Watch out") |

Want the raw output behind a row? Ask the router directly:

In [ ]:
# R1: any single read-only command, real output
r1('show ip ssh | include KEX|Hostkey|version');

### Automation

`verify.yml` only reads (NETCONF and `show`), so it's safe at any time. It proves
the playbooks can reach all three routers and prints their crypto posture.

In [ ]:
# Ansible: read-only posture report across r1, r2, r3
playbook('verify.yml');

**`Could not open socket to <router>:830`?** NETCONF isn't on. `bootstrap.yml`
turns it on (that's a config change, so it's not part of the read-only checks)
and waits for port 830 to answer. Run it, then rerun `verify.yml`.

In [ ]:
# Ansible: enable NETCONF on r1, r2, r3 (only if verify.yml can't reach port 830)
playbook('ansible-playbook bootstrap.yml');

---
<a id="act1"></a>
## Act 1: learn it in containers

An OpenSSH 10.4 server and client on one Docker network, with `tcpdump` on the
server. One cell per action: read the comment, run it, read the output.

### Demo 1: hybrid key exchange by default, and what happens without it

What does modern OpenSSH do with zero config? Look for:

- `kex: algorithm: mlkem768x25519-sha256` with no KEX setting anywhere
- the KEX init and reply: **1304 B** and **1716 B** on the wire
- a classical-only server: the client connects anyway, with a loud `** WARNING`
  that the connection isn't post-quantum
- (optional) composite `ssh-mldsa44-ed25519` keys, so authentication is
  post-quantum too: a 1383 B host key and a 2523 B signature

Keep the first line in mind. Act 2 asks whether a router does the same.

**Beat 1: start sshd, connect with no KEX config**

In [ ]:
# SSH server: start sshd with the shipped config (hybrid KEX by default)
ssh_server("pkill -f [s]shd 2>/dev/null; sleep 1; true")
ssh_server("/usr/local/sbin/sshd -D -e -f /cfg/sshd_config > /tmp/sshd.log 2>&1 &");

In [ ]:
# SSH client: zero KEX config; show the negotiated algorithm
kex = ssh_client("ssh -F /cfg/ssh_config -v ssh-server 'echo AUTH_OK' 2>&1 | grep -E 'kex: algorithm|Authenticated to|AUTH_OK'")

In [ ]:
panel(ssh.stage, step=4, focus=["KEX init", "KEX reply"], status=("ok", "ok"),
      rows=[("Key-exchange algorithm", "mlkem768x25519-sha256",
             (grep(kex.text, 'kex: algorithm') or ['?'])[0], 'mlkem768x25519' in kex.text)],
      notes=[("Client / server KEX bytes", "1304 B / 1716 B"), ("Round trips", "1 (same as classical)")],
      banner="Modern OpenSSH negotiates hybrid ML-KEM with zero configuration.")

**Beat 2: model an un-upgraded server (classical-only KEX): LOUD warning**

In [ ]:
# SSH server: restart offering only a classical KEX
ssh_server("pkill -f [s]shd 2>/dev/null; sleep 1; true")
ssh_server("/usr/local/sbin/sshd -D -e -f /cfg/sshd_config -o KexAlgorithms=curve25519-sha256 > /tmp/sshd.log 2>&1 &");

In [ ]:
# SSH client still offers hybrid, is forced down, and warns
warn = ssh_client("ssh -F /cfg/ssh_config ssh-server 'true' 2>&1 | grep -iE 'WARNING|post-quantum|store now' || true")

In [ ]:
# The KEX init/reply arrows below still say '1304 B'/'1716 B' from Beat 1; retag
# them for this downgraded run, and bold the KEXINIT negotiation where it happens
ssh.stage.retag("KEX init", "x25519 only, no ML-KEM", direction="r")
ssh.stage.retag("KEX reply", "x25519 only, no ML-KEM", direction="l")
panel(ssh.stage, step=1, focus="KEXINIT", status=("warn", "warn"),
      rows=[("Client downgrade warning", "** WARNING ...", warn.text or "(none)", bool(warn.text.strip()))],
      banner="SSH downgrade is LOUD: the client warns the session is not post-quantum. Contrast the silent MACsec TLS 1.2 fallback above.",
      banner_color=ORANGE)

**Beat 3: restore the PQ-capable default**

In [ ]:
# SSH server: restart with the shipped (hybrid) config
ssh_server("pkill -f [s]shd 2>/dev/null; sleep 1; true")
ssh_server("/usr/local/sbin/sshd -D -e -f /cfg/sshd_config > /tmp/sshd.log 2>&1 &")
print('sshd restored to the hybrid default')
# Restore the KEX arrow tags now that the server is hybrid again
ssh.stage.retag("KEX init", "1304 B", direction="r")
ssh.stage.retag("KEX reply", "1716 B", direction="l")

**Beat 4 (optional): composite ML-DSA-44 + Ed25519 authentication (both pillars PQ)**

In [ ]:
# host: reissue host + user keys as composite mldsa44-ed25519
host("cd learn/ssh && docker compose run --rm keygen mldsa44-ed25519");

In [ ]:
# SSH server: offer the composite host key, start a capture
ssh_server("pkill -f [s]shd 2>/dev/null; pkill -f [t]cpdump 2>/dev/null; sleep 1; true")
ssh_server("/usr/local/sbin/sshd -D -e -f /cfg/sshd_config -o HostKeyAlgorithms=ssh-mldsa44-ed25519@openssh.com -o PubkeyAcceptedAlgorithms=ssh-mldsa44-ed25519@openssh.com > /tmp/sshd.log 2>&1 &")
ssh_server("tcpdump -i any -U 'tcp port 22' -w /tmp/comp.pcap &");

In [ ]:
# SSH client: authenticate with the composite type
auth = ssh_client("ssh -F /cfg/ssh_config -o HostKeyAlgorithms=ssh-mldsa44-ed25519@openssh.com -o PubkeyAcceptedAlgorithms=ssh-mldsa44-ed25519@openssh.com -v ssh-server 'echo PQ_AUTH_OK' 2>&1 | grep -iE 'host key algorithm|Server accepts key|PQ_AUTH_OK'")

In [ ]:
# SSH server: tcpdump needs a moment to drain, then read the cleartext sizes
ssh_server("sleep 2; pkill -f [t]cpdump 2>/dev/null; sleep 1; true")
sizes = ssh_server("tshark -r /tmp/comp.pcap -V 2>/dev/null | grep -iE 'Host key length|Host signature length'")
hostkey = first_int((grep(sizes.text, 'Host key length') or [''])[0])
hostsig = first_int((grep(sizes.text, 'Host signature length') or [''])[0])

In [ ]:
# The 'KEX reply' arrow is where the host key + its signature actually ride,
# retag it with what we just measured, and bold it (not the auth arrow after it)
ssh.stage.retag("KEX reply", f"host key {hostkey} B + sig {hostsig} B", direction="l")
panel(ssh.stage, step=4, focus="KEX reply", status=("ok", "ok"),
      rows=[("Composite host key", "1383 B", hostkey + ' B', hostkey == '1383'),
            ("Composite host signature", "2523 B", hostsig + ' B', hostsig == '2523')],
      banner="Both pillars post-quantum: hybrid ML-KEM key exchange + composite ML-DSA-44 authentication (opt-in).",
      banner_color=MAGENTA)

In [ ]:
# reset (stop sshd + capture)
ssh_server("pkill -f [s]shd 2>/dev/null; pkill -f [t]cpdump 2>/dev/null; true");

Something stuck? This stops stray `sshd` and capture processes without taking
the stack down:

In [ ]:
# host: stop stray sshd / tcpdump processes, keep the stack up
labs.panic()

---
<a id="act2-setup"></a>
## Starting at Act 2? Set up the routers

Skipping the containers? Start here. This cell loads its own helpers, so you
don't need the Setup cell at the top. Then, on each router:

1. removes any `ip ssh server algorithm kex` line, back to the platform default
2. leaves `ip ssh server algorithm hostkey` alone: if you pinned it, it's protecting you
3. NETCONF on all three, for Act 3
4. the same read-only check as the pre-flight

**This cell configures the routers**, unlike every other router cell here. On a
lab that's already in shape it only reads.

In [ ]:
# R1, R2, R3: CONFIGURES the Act 2 starting state, then checks it
# (loads the helpers first, so this works as the first cell you run)
import sys, pathlib
_here = pathlib.Path.cwd()
_demos = next((p for p in (_here.parent, _here, _here / 'demos') if (p / 'common' / 'lab.py').exists()), None)
if _demos is None:
    raise RuntimeError('Start Jupyter inside the repo (see demos/README.md) so the helpers can be found.')
for _p in ('containers_demo', 'common', 'ssh_demo'):
    sys.path.insert(0, str(_demos / _p))
from ssh_lab import *  # noqa: F401,F403  routers, playbooks, probes

setup()

---
<a id="act2"></a>
## Act 2: the router's SSH server, from your laptop

**You type every router command in this act at the R1 CLI.** The probes connect
from your laptop to R1's management address and stop after the key exchange, so
what you see is what your own `ssh admin@R1` gets.

**Keep your current R1 session open** until a new one works. This act changes the
crypto on the very connection you manage the box over.

### Demo 2: what does the router offer today?

**R1:**

```
show ip ssh | include KEX
```

That's the platform default: `curve25519-sha256` first, then ECDH and DH groups.
No ML-KEM. Now ask from the laptop:

In [ ]:
# laptop -> R1:22: what do you and the router agree on today?
ssh_kex_probe('r1', expect=None, expect_offer=False);

Classical, whatever your client prefers, because the router doesn't offer anything
else. Read the `client offers` line too: your client already speaks
`mlkem768x25519-sha256`. The router is the one holding it back.

### Demo 3: one line on the router

Configure the SSH server to offer ML-KEM first. The order is the server's
preference; the classical entries are the fallback for clients that have never
heard of ML-KEM. **R1:**

```
conf t
 ip ssh server algorithm kex mlkem768x25519-sha256 curve25519-sha256 ecdh-sha2-nistp256
end
show ip ssh | include KEX
```

Expect `KEX Algorithms:mlkem768x25519-sha256,curve25519-sha256,ecdh-sha2-nistp256`.
26.2 has three hybrids (`mlkem768x25519-sha256`, `mlkem768nistp256-sha256`,
`mlkem1024nistp384-sha384`) and no pure ML-KEM: a break in either half still
leaves you the other.

In [ ]:
# R1: ML-KEM is offered first (read-only)
router_check('Demo 3: ML-KEM offered', 'R1', [
    Check('r1', 'show ip ssh', 'KEX offer order', 'mlkem768x25519-sha256 first', r'KEX Algorithms:mlkem768x25519-sha256,'),
    Check('r1', 'show running-config | include ^ip ssh server algorithm', 'config line', 'ip ssh server algorithm kex mlkem768x25519-sha256 ...', r'algorithm kex mlkem768x25519-sha256'),
])

Now connect again, exactly as you did in Demo 2:

In [ ]:
# laptop -> R1:22: same client, the router now offers ML-KEM first
ssh_kex_probe('r1', expect=None, expect_offer=True);

Did you get `mlkem768x25519-sha256`? **On a Mac, probably not.** In SSH the
*client's* order decides: the first algorithm on the client's list that the
server also supports wins. macOS ships `/etc/ssh/crypto.conf` with
`KexAlgorithms ^ecdh-sha2-nistp256`, which moves P-256 to the front of every
connection's list. Look at `client's first choice` in the panel: the router
offers ML-KEM first, and your client still picks P-256.

On a Linux client with OpenSSH 9.9 or newer, and no system crypto policy
reordering the list, the same probe gives ML-KEM straight away. Two ways to see
it from here: OpenSSH's own defaults (`-F none` skips every config file), or
pinning the algorithm the way `ssh.md` does.

In [ ]:
# laptop -> R1:22: OpenSSH's built-in defaults, no config files
ssh_kex_probe('r1', clean_config=True);

In [ ]:
# laptop -> R1:22: ask for ML-KEM explicitly (ssh -o KexAlgorithms=...)
ssh_kex_probe('r1', kex='mlkem768x25519-sha256');

Want it in your everyday `ssh`? Add `KexAlgorithms mlkem768x25519-sha256,` plus
your fallbacks to the router's `Host` entry in `~/.ssh/config`. That's the
real-world lesson here: **upgrading the server is half the job; the client's
policy decides what you actually get.**

### Demo 4: the host key is still classical

Every probe above said `host key algorithm: rsa-sha2-512`. The key exchange is
post-quantum now; the router's proof of identity isn't. Can it be? **R1:**

```
conf t
 ip ssh server algorithm hostkey ?
end
```

RSA and ECDSA, nothing else: no ML-DSA, no composite. Same for user keys
(`ip ssh server algorithm publickey ?`). That's fine for now. Key exchange
protects against *harvest now, decrypt later*, which starts the day someone
records your session. Forging a host key needs a quantum computer in real time.

**Put R1 back** before Act 3, so the playbook starts from the default too:

```
conf t
 no ip ssh server algorithm kex
end
```

In [ ]:
# R1, R2, R3: back in the starting state (read-only)
check()

---
<a id="act3"></a>
## Act 3: now do it to the fabric

One router by hand is fine. What about fifty? `ssh-pq.yml` pushes the KEX list
and a host key pin to every router as one NETCONF edit. The list comes from
`pq_ssh_kex` in `group_vars/all/pqc.yml`: `mlkem768x25519-sha256`,
`mlkem768nistp256-sha256`, `curve25519-sha256`, `ecdh-sha2-nistp256`. The pin is
`rsa-sha2-512 rsa-sha2-256`, the fix for the EC host key trap.

It's the one playbook that changes the crypto on the connection it manages the
box over, so it refuses a list with no classical fallback unless you pass
`-e ssh_pq_allow_no_fallback=true`.

### Demo 5: Ansible offers ML-KEM everywhere

Run it twice. The first run changes every router; the second finds nothing to
do.

Captured run (`captured/08-ssh-pq/`): `changed=1` on r2 and r3, then `changed=0`.
(R1 already had the line from the hand exercise when that was captured; you put
it back, so expect `changed=1` on all three.) Each run takes about 37 s.

Why can the panel say 1 when the recap says 2? Every playbook ends with `verify.yml`, whose posture report is forced to `changed` so it stays on screen. The panel subtracts it and counts config changes only, which is what the captured runs show.

These cells keep the `PLAY RECAP` on screen. **Heads up:** in a plain terminal the repo's `pqc_lab` callback hides the recap when a run succeeds. Run them from here, or prefix `ANSIBLE_STDOUT_CALLBACK=default` in a terminal (plus `-e @../../../demos/common/lab.yml`; see [Running a playbook from a terminal](../README.md#run-a-playbook-from-a-terminal)).

In [ ]:
# Ansible: KEX list + host key pin on r1, r2, r3
playbook('ansible-playbook ssh-pq.yml', expect_changed=1);

In [ ]:
# Ansible: same playbook again; nothing to do is the point
playbook('ansible-playbook ssh-pq.yml', expect_changed=0);

In [ ]:
# laptop -> R1, R2, R3: each router offers ML-KEM; your config, then pinned
probe_all();

Same lesson as Demo 3, fleet-wide: every router offers ML-KEM, your client's
policy decides whether you use it, and pinning it proves every box will.

---
<a id="after"></a>
## Clean up

Teardown removes **both** lists, the KEX order and the host key pin, so each router
goes back to its platform default. Pinned the host key yourself before this demo?
Put it back afterwards.

In [ ]:
# Ansible: remove the KEX list and host key pin from r1, r2, r3
playbook('ansible-playbook ssh-pq.yml -e state=absent', expect_changed=1);

In [ ]:
# R1, R2, R3: back in the starting state; nobody offers ML-KEM
check() and probe_all(expect_offer=False);

On the laptop: container down, generated keys removed, `learn/ssh` back to the
committed state.

In [ ]:
# host: Act 1 stacks down, generated keys and certs removed, learn/ssh back to the committed state
labs.cleanup()

**Run it again?** Start from [Before each run](#preflight). Clean up leaves the
routers in the starting state, so `check()` should be green straight away.